# 03 Transfer from the z-stack to the Andor Zyla flow dataset

The CNN (64 × 64), the chip network (8 × 8) and the feature control are trained on the whole z-stack dataset
(static cells) and applied to all images of the Andor Zyla flow dataset. 20 % of the z-stack stacks are used for
validation (epoch selection). The flow labels are used for the evaluation only. Both datasets are resampled to
0.65 µm/px.

Thesis: Results, Section "Classification of the Andor Zyla flow dataset", paragraph "Transfer from the static
z-stack" (Table `tab:res-transfer`, column "Z-stack to flow", and the raw contrast values in the text).

Output: `results/03_zstack_to_zyla_flow_metrics.csv` and `_predictions.csv.gz`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

## 1. Data
The edge check removes Zyla flow crops whose square crop window reached beyond the camera ROI (see notebook 01).

In [ ]:
data = C.load_data(sets=['zstack', 'flow_zyla'])
Z, F = data['zstack'], data['flow_zyla']
print('Andor Zyla flow recordings per cell line:', F.meta.groupby('label').group.nunique().to_dict())

## 2. Training on the z-stack, test on the Andor Zyla flow dataset

In [ ]:
targets = {'flow_zyla': (F.meta, F.X)}
metrics, preds = [], []
for kind, size in C.MODELS:
    m, p = M.run_transfer(Z.meta, Z.X, targets, kind, size, 'group_stack', C.TRAIN, C.SEEDS_BY_MODEL[kind],
                          C.VAL_FRAC)
    metrics.append(m); preds.append(p)
metrics.append(M.control_transfer(Z.meta, Z.feat, {'flow_zyla': (F.meta, F.feat)}))
metrics = pd.concat(metrics, ignore_index=True)
preds = pd.concat(preds, ignore_index=True)
M.save_results('03_zstack_to_zyla_flow', C.RESULTS_DIR, metrics, preds)

## 3. Results
Macro-F1: mean and range over the three seeds (the feature control has no seeds). Recall per class: fraction of
correctly classified images, from the predictions of all seeds.

In [ ]:
table = metrics.groupby('model').macro_f1.agg(['mean', 'min', 'max', 'count'])
display(table.round(3))
display(preds.groupby(['model', 'label']).correct.mean().unstack('label').round(3))

### Raw contrast
Median of the raw contrast (standard deviation divided by the mean intensity, before normalization) per dataset and
cell line. This is one of the four inputs of the feature control.

In [ ]:
contrast = pd.concat([d.feat.assign(dataset=k, label=d.meta.label.to_numpy()) for k, d in data.items()])
contrast['contrast'] = contrast.raw_std / contrast.raw_mean.abs().clip(lower=1e-6)
display(contrast.groupby(['dataset', 'label']).contrast.median().unstack('label').round(3))